# Building a chatbot finetune on Qwen/Qwen3-4B

In [2]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')

!git clone https://{token}@github.com/nando0307/ChatWithMe.git
%cd ChatWithMe
!git config user.name "nando"
!git config user.email "lenando0307@gmail.com"

Cloning into 'ChatWithMe'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.
/content/ChatWithMe


## 1. Install and Import

In [3]:
!pip install -q accelerate peft bitsandbytes transformers>=4.51.0 trl datasets matplotlib

import torch
import math
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
from datasets import load_dataset


## 2. Data Preprocessing

In [5]:
# Load dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(f'Total examples: {len(dataset)}')

Total examples: 51760


In [6]:
print(dataset[0])

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [7]:
# Format into ChatML
def format_chatml(example):
    instruction = example["instruction"]
    if example["input"]:
        instruction = f"{instruction}\n{example['input']}"

    text = (
        f"<|im_start|>user\n{instruction}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": text}

In [8]:
dataset = dataset.map(format_chatml, remove_columns=dataset.column_names)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [9]:
# Split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\nSample:\n{train_dataset[0]['text']}")

Train: 46584 | Val: 5176

Sample:
<|im_start|>user
Analyze the user's writing style in the given text snippet and provide 3 suggestions to improve it.
Weather now bad outside also it is cold. People no have fun time, always grumble because of cold weather. Use warm clothes protect from freezing.<|im_end|>
<|im_start|>assistant
1. Improve grammar and sentence structure: The sentences are fragmented and lack proper grammar. Consider rewriting them as, 'The weather outside is bad, and it's cold. People are not having a good time, and they are always grumbling because of the cold weather. Wear warm clothes to protect yourself from freezing.' 
2. Use more descriptive words: Replace 'bad' and 'cold' with more specific and descriptive words like 'gloomy' or 'frigid.'
3. Make the text flow smoothly by using conjunctions and transitions: Use words like 'however,' 'moreover,' and 'therefore' to connect ideas and make the text easier to read.<|im_end|>


In [10]:
# Commit changes after Data Preprocessing
!git add .
!git commit -m "Data Preprocessing: Load Alpaca dataset, format to ChatML, and split data"
!git push origin main

[main 632a63e] Data Preprocessing: Load Alpaca dataset, format to ChatML, and split data
 1 file changed, 4 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 405 bytes | 405.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/nando0307/ChatWithMe.git
   dd3fc35..632a63e  main -> main


In [16]:
# Save notebook to Drive first
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
!ls "/content/drive/MyDrive/Colab Notebooks/"

 ASL_detect.ipynb	        Untitled	       Untitled2-2.ipynb
 ChatWithMe.ipynb	        Untitled0.ipynb        Untitled2.ipynb
'flower_classifier (1).ipynb'   Untitled1.ipynb
 flower_classifier.ipynb       'Untitled2 (1).ipynb'


In [ ]:
# Copy it into your repo
!cp "/content/drive/MyDrive/Colab Notebooks/ChatWithMe.ipynb" /content/ChatWithMe/

## 3. Model Initialization

## 4. Model Training

## 5. Visualization of Metrics and Loss

## 6. Model Evaluation

## 7.  Model Inference

## 8. Merge & Push to HuggingFace Hub